# Heart Rate & Stress Analysis

Visualize intraday heart rate patterns, resting HR trends, and daily stress/recovery balance.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from oura_client import OuraClient

client = OuraClient(sandbox=True)

START = "2025-01-01T00:00:00"
END = "2025-01-08T00:00:00"
START_DATE = "2025-01-01"
END_DATE = "2025-03-01"

## Fetch Heart Rate Time Series

In [ ]:
hr_data = client.get_heart_rate(start_datetime=START, end_datetime=END)
print(f"Fetched {len(hr_data)} heart rate samples")

hr_df = pd.DataFrame([
    {"timestamp": h.timestamp, "bpm": h.bpm, "source": h.source.value}
    for h in hr_data
])
if not hr_df.empty:
    hr_df["timestamp"] = pd.to_datetime(hr_df["timestamp"])
    hr_df = hr_df.sort_values("timestamp")
hr_df.head(10)

## Intraday Heart Rate

In [ ]:
if not hr_df.empty:
    fig = px.scatter(
        hr_df, x="timestamp", y="bpm", color="source",
        title="Heart Rate Throughout the Day (colored by source)",
        labels={"bpm": "BPM", "timestamp": "Time"},
        template="plotly_white",
        opacity=0.7,
    )
    fig.update_traces(marker=dict(size=4))
    fig.show()
else:
    print("No heart rate data available.")

## Heart Rate Distribution by Source

In [ ]:
if not hr_df.empty:
    fig = px.box(
        hr_df, x="source", y="bpm", color="source",
        title="Heart Rate Distribution by Source",
        template="plotly_white",
    )
    fig.show()
else:
    print("No heart rate data available.")

## Fetch Daily Stress Data

In [ ]:
stress_data = client.get_daily_stress(start_date=START_DATE, end_date=END_DATE)
print(f"Fetched {len(stress_data)} daily stress records")

stress_df = pd.DataFrame([
    {
        "day": s.day,
        "stress_min": s.stress_high_minutes,
        "recovery_min": s.recovery_high_minutes,
        "summary": s.day_summary.value if s.day_summary else None,
    }
    for s in stress_data
]).sort_values("day").reset_index(drop=True)

stress_df.head(10)

## Stress vs Recovery Balance

In [ ]:
if not stress_df.empty and stress_df[["stress_min", "recovery_min"]].notna().any().any():
    fig = go.Figure()
    fig.add_trace(go.Bar(x=stress_df["day"], y=stress_df["stress_min"], name="High Stress (min)", marker_color="#ef5350"))
    fig.add_trace(go.Bar(x=stress_df["day"], y=stress_df["recovery_min"], name="High Recovery (min)", marker_color="#66bb6a"))
    fig.update_layout(
        barmode="group",
        title="Daily Stress vs Recovery Time",
        xaxis_title="Date",
        yaxis_title="Minutes",
        template="plotly_white",
    )
    fig.show()
else:
    print("No stress data available.")

## Stress Summary Distribution

In [ ]:
summary_counts = stress_df["summary"].dropna().value_counts()
if not summary_counts.empty:
    fig = px.pie(
        values=summary_counts.values,
        names=summary_counts.index,
        title="Day Summary Distribution",
        color=summary_counts.index,
        color_discrete_map={"restored": "#66bb6a", "normal": "#42a5f5", "stressful": "#ef5350"},
        template="plotly_white",
    )
    fig.show()
else:
    print("No summary data available.")

## Resilience Trend

In [ ]:
resilience_data = client.get_daily_resilience(start_date=START_DATE, end_date=END_DATE)
print(f"Fetched {len(resilience_data)} resilience records")

if resilience_data:
    res_df = pd.DataFrame([
        {
            "day": r.day,
            "level": r.level.value,
            "sleep_recovery": r.contributors.sleep_recovery,
            "daytime_recovery": r.contributors.daytime_recovery,
            "stress": r.contributors.stress,
        }
        for r in resilience_data
    ]).sort_values("day")

    level_order = ["limited", "adequate", "solid", "strong", "exceptional"]
    res_df["level_num"] = res_df["level"].map({v: i for i, v in enumerate(level_order)})

    fig = px.scatter(
        res_df, x="day", y="level_num", color="level",
        title="Resilience Level Over Time",
        labels={"level_num": "Level", "day": "Date"},
        template="plotly_white",
        category_orders={"level": level_order},
    )
    fig.update_yaxes(tickvals=list(range(5)), ticktext=level_order)
    fig.show()
else:
    print("No resilience data available.")